## Project 1: Blend Learning for Fraud Detection

A transaction is scored by an MLP, a Q-learning agent moves the alert threshold, and a rule table can override that score.

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader, TensorDataset

In [ ]:
# Make results reproducible
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
DATA_PATH = "creditcard.csv"
# 0 = legitimate, 1 = fraud
FRAUD_LABEL = 1
LEARNING_RATE = 0.001
EPOCHS = 15
BATCH_SIZE = 256
HIDDEN_DIM = 16
# Alert if the MLP score is at least this value
BASE_THRESHOLD = 0.5
# Symbolic limits used by the rule table
AMOUNT_LIMIT = 1000.0
VELOCITY_WINDOW = 10
VELOCITY_LIMIT = 5
# Holdout is the last fraction of rows, kept in time order
HOLDOUT_FRACTION = 0.2

In [ ]:
def load_transactions(path=DATA_PATH):
  # TODO: read creditcard.csv
  # TODO: print shape, fraud count, and fraud rate
  pass

In [ ]:
def scale_and_split(frame):
  # Scale Time and Amount with the training mean and standard deviation
  # Keep the last HOLDOUT_FRACTION of rows in time order for the agent
  # TODO: return X_train, y_train, X_holdout, y_holdout, holdout_frame
  pass

## Connectionist (Deep Learning)

MLP scores each transaction. Same two-layer shape as the Ex1 network, from Lecture 2 Slides 93 and 94.

In [ ]:
# MLP from the connectionist component
# Perceptron-style classifier from Lecture 2 Slide 46,
# extended to a multi-layer net from Lecture 2 Slides 93 and 94
# features -> hidden -> fraud score
class FraudMLP(nn.Module):
  def __init__(self, num_inputs, hidden_dim=HIDDEN_DIM):
    super().__init__()
    self.hidden = nn.Linear(num_inputs, hidden_dim)
    self.output = nn.Linear(hidden_dim, 1)

  def forward(self, inputs):
    hidden = torch.relu(self.hidden(inputs))
    score = torch.sigmoid(self.output(hidden))
    return score

In [ ]:
def train_fraud_mlp(X_train, y_train):
  # Training loop inspired by Lecture 2 Slides 93 and 94
  # TODO: build FraudMLP
  # TODO: BCELoss with a higher weight on fraud
  # TODO: print one line per epoch
  # Epoch {epoch:5d} | Loss: {loss:.4f} | Precision: {precision:.2f} | Recall: {recall:.2f} | F1: {f1:.2f}
  pass

In [ ]:
def fraud_score(model, inputs):
  # Return a probability in [0, 1]
  model.eval()
  with torch.no_grad():
    score = model(inputs)
  return score

## Intuitive (Reinforcement Learning)

The agent watches the holdout stream and moves the alert threshold when the fraud rate shifts.

In [ ]:
# Q-learning agent for the intuitive component
# Adjusts the alert threshold when the fraud rate shifts
class ThresholdAgent:
  def __init__(self, learning_rate=0.1, discount=0.9, base_threshold=BASE_THRESHOLD):
    self.learning_rate = learning_rate
    self.discount = discount
    self.threshold = base_threshold
    self.q_table = {}

  def current_threshold(self):
    return self.threshold

  def choose_action(self, state):
    # TODO: lower, hold, or raise the threshold
    pass

  def update(self, state, action, reward, next_state):
    # TODO: Q-learning update
    # Reward: +1 caught fraud, -1 false alert, 0 otherwise
    pass

In [ ]:
def run_threshold_agent(model, agent, X_holdout, y_holdout):
  # TODO: walk the holdout in time order
  # TODO: state is recent fraud rate and whether the last alert was a false positive
  # TODO: print the threshold at the start and at the end
  # TODO: return scores and the threshold history for the plot
  pass

## Symbolic (Rule-Based Logic)

First matching rule wins. `V1`–`V28` stay out of the rule table because they are PCA features.

In [ ]:
# Rule table for the symbolic component
# First matching rule decides the audit label
def apply_rules(amount, recent_count, score, threshold):
  if amount >= AMOUNT_LIMIT:
    return "review", "amount_limit"
  if recent_count >= VELOCITY_LIMIT:
    return "review", "velocity_limit"
  if score >= threshold:
    return "alert", "score_above_threshold"
  return "allow", "none"

In [ ]:
def blend_decision(model, agent, features, amount, recent_count):
  # TODO: score = fraud_score(model, features)
  # TODO: threshold = agent.current_threshold()
  # TODO: decision, rule = apply_rules(amount, recent_count, score, threshold)
  # TODO: return score, threshold, rule, decision
  pass

## Testing for Fraud Detection

In [ ]:
# Four hand-built cases, same print layout as the Ex1 gate table
test_cases = [
  {"amount": 10.0, "recent_count": 1, "score": 0.1, "threshold": BASE_THRESHOLD},
  {"amount": 40.0, "recent_count": 1, "score": 0.9, "threshold": BASE_THRESHOLD},
  {"amount": AMOUNT_LIMIT, "recent_count": 1, "score": 0.1, "threshold": BASE_THRESHOLD},
  {"amount": 40.0, "recent_count": VELOCITY_LIMIT, "score": 0.1, "threshold": BASE_THRESHOLD},
]

print("amount\t|recent\t|score\t|threshold\t|rule\t|decision")
for case in test_cases:
  decision, rule = apply_rules(
    case["amount"],
    case["recent_count"],
    case["score"],
    case["threshold"]
  )
  print(
    f"{case['amount']}\t|{case['recent_count']}\t|"
    f"{case['score']}\t|{case['threshold']}\t|"
    f"{rule}\t|{decision}"
  )

In [ ]:
def holdout_metrics(y_true, decisions):
  # TODO: precision, recall, F1
  # TODO: print a 2x2 confusion matrix
  pass

In [ ]:
def plot_threshold(history):
  # Plotting function
  # Inspired by code from Lecture 1 Slides 155-157
  # TODO: plot the threshold over the holdout stream
  pass

In [ ]:
def print_audit(rows):
  # TODO: print 10 holdout rows
  # time | amount | score | threshold | rule | decision
  print("time\t|amount\t|score\t|threshold\t|rule\t|decision")